<a href="https://colab.research.google.com/github/Figueroa1222/taller-calidad-datosGestionDatosG3/blob/main/taller3_calidad_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller 3 - Gestión de datos
## Integrantes:
Anastasia Minina

Cristhian Burbano

Denis Chancay

Erick Figueroa

## Paso 1: Importación de librerías
Uso de las principales librerías para tratamiento, exploración y limpieza de datos más usadas en la actualidad.

In [ ]:
# Instalación de librerías adicionales (no vienen preinstaladas en Colab)
!pip install pandera ydata-profiling missingno -q
import pandas as pd
import numpy as np
import missingno as msno
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
 MinMaxScaler, RobustScaler, MaxAbsScaler, StandardScaler,
 PowerTransformer, Normalizer,
)
import pandera.pandas as pa
from pandera import Column, Check, DataFrameSchema

## Paso 2: Descargar y cargar el dataset
Se uso del dataset Cafe Sales — Dirty Data for Cleaning Training, el mismo fue obtenido desde la plataforma Kaggle. Y vemos los 5 primeros registros para una observación inicial del dataset y lo que contiene.

In [ ]:
df = pd.read_csv("dirty_cafe_sales.csv")
df.head()

## Paso 3: Exploración inicial y perfilado de datos
Con la función info() obtenemos una descripción inicial del tipo de datos en cada columna y las dimensiones del dataset, tanto en filas y columnas. la función describe() nos muestra un análisis estadístico descriptivo de varibles cualitativas y cuantitativas.

In [ ]:
df.info()
df.describe(include="all")

La función matrix() de la libreria missingno visualiza como se están distribuido los datos faltantes en cada columna, permitiendo encontrar patrónes que nos ayudan a comprender mejor los datos y decidir si aplicar o no técnicas de imputación.

In [ ]:
msno.matrix(df)

La función ydata-profiling genera un reporte de calidad e interactivo de los factores que se deben considerar para el análisis.

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Reporte de calidad — Tech Coffee Shop", minimal=True)
profile.to_notebook_iframe()

 ## Paso 4: Tratamiento de datos faltantes
 Es importante para un análisis de calidad tratar los valores nulos que en la mayoría de las bases de datos, adminten campos nulos. Con la función isnull() comprobamos eso, a la vez, que contamos los valores nulos por campo presentes en nuestro dataset.

In [ ]:
missing_values_count = df.isnull().sum()
missing_values_count

Saber en que medida los datos nulos están presentes en nuestros datos es importante, ya que esa medida escalar nos ayuda a decidir si se realizan procesos de eliminación e imputación. Se observa que en 2 campos el porcentaje es considerablemente alto.

In [ ]:
(missing_values_count / len(df) * 100).round(2)

Eliminar datos arbitariamente solo por tener datos nulos, sin considerar como afecta a nuestro resultado final no es el adecuado, pero para fines práticos y cuando se necesite; la instrucción a continuación muestra como se haría la eliminación de datos nulos.

In [ ]:
df_sin_filas_nulas = df.dropna()
print(f"Filas originales: {len(df)}")
print(f"Filas después de eliminar nulos: {len(df_sin_filas_nulas)}")
print(f"Filas perdidas: {(1 - len(df_sin_filas_nulas) / len(df)):.1%}")

Al igual que los registros, eliminar columnas sin considerar lo antes mencionado no es buena prática en el proceso de limpieza de datos. En este caso, elimina columnas en donde almenos exista 1 datos nulo.

In [ ]:
df_sin_columnas_nulas = df.dropna(axis=1)
columnas_originales = df.shape[1]
columnas_restantes = df_sin_columnas_nulas.shape[1]
print(f"Columnas originales: {columnas_originales}")
print(f"Columnas restantes: {columnas_restantes}")
print(f"Columnas eliminadas: {columnas_originales - columnas_restantes}")

El proceso correcto cuando se tiene alto porcentaje de valores faltantes es la imputación (inserción de valores obtenidos estadísticamente). Se debe comprobar que los tipos de datos sean los mismos, ya que puede haber mezcla de datos, lo correctp es convertir a numéricos. Para "rellenar los huecos" en nuestro dataset de prática usamos la función bfill() y strategy="median" reemplazo esos "huecos" con la mediana de la columna.

In [ ]:
df["Total Spent"].unique()
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")
df["Total Spent"].unique()
df["Total Spent"] = df["Total Spent"].bfill()
df["Total Spent"].isnull().sum()
imputer = SimpleImputer(strategy="median")
df[["Total Spent"]] = imputer.fit_transform(df[["Total Spent"]])
df[["Total Spent"]].head(30)

## Paso 5: Corrección de inconsistencias de tipo y valores atípicos
Corregir los valores de tipo no numérica en las columnas numéricas faltantes.

In [ ]:
columnas_numericas = ["Quantity", "Price Per Unit", "Total Spent"]
for columna in columnas_numericas:
 df[columna] = pd.to_numeric(df[columna], errors="coerce")
df[columnas_numericas].dtypes

## Paso 6: Escalado y normalización de datos
Para el proceso de limpieza de datos, es importante escalar y normalizar el conjunto de datos. al escalar estamos ajustando los datos y normalizar ajustamos la distribución de los mismos, el código a continuación reliza un escalado al campo Price Per Unit.

In [ ]:
precio_original = df[["Price Per Unit"]].copy()
scaler = MinMaxScaler()
df["Price Per Unit_scaled"] = scaler.fit_transform(df[["Price Per Unit"]])
print("Original -> min:", precio_original.min().values[0], " max:",
precio_original.max().values[0])
print("Escalado -> min:", df["Price Per Unit_scaled"].min(), " max:", df["Price Per Unit_scaled"].max())

Existen varias técnicas para realizar escalado como: MinMaxScaler,RobustScaler y MaxAbsScaler. La elección de estas depende del comportamiento de los datos y lo que se desee obtener.

In [ ]:
robust_scaler = RobustScaler()
df["Price Per Unit_robust"] = robust_scaler.fit_transform(df[["Price Per Unit"]])
print("Original -> min:", precio_original.min().values[0], " max:",
precio_original.max().values[0])
print("Escalado -> min:", df["Price Per Unit_robust"].min(), " max:", df["Price Per Unit_robust"].max())

In [ ]:
maxabs_scaler = MaxAbsScaler()
df["Price Per Unit_maxabs"] = maxabs_scaler.fit_transform(df[["Price Per Unit"]])
print("Original -> min:", precio_original.min().values[0], " max:",
precio_original.max().values[0])
print("Escalado -> min:", df["Price Per Unit_maxabs"].min(), " max:", df["Price Per Unit_maxabs"].max())

Un paso importante antes de realizar la normalización, se debe tener en cuenta que todos los tipos de datos sean similares, para estos casos de tipo numérico, ya que lo calculos estadístico no pueden realizar correctamente sus operaciones con tipos de texto.

In [ ]:
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
pt = PowerTransformer(method="box-cox")
df["Price Per Unit_boxcox"] = pt.fit_transform(df[["Price Per Unit"]])


Al igual que con el escalado, existen más técnicas de normalización que llevan los datos a una distribución con la media o desviación estándar. Para este caso, se usó la función Normalizer() el cual con el método l1 ajusta las filas para la suma de los valores absolutos sea 1; y l2, concurre en la suma de los cuadrados sea 1.

In [ ]:
##std_scaler = StandardScaler()
##df["Price Per Unit_standard"] = std_scaler.fit_transform(df[["Price Per Unit"]])
filas_validas = df[["Quantity", "Price Per Unit"]].dropna()
l2_normalizer = Normalizer(norm="l2")
df.loc[filas_validas.index, ["Quantity_l2", "Price Per Unit_l2"]] = (l2_normalizer.fit_transform(filas_validas))

## Paso 7: Validación automatizada de calidad de datos
En este paso, validamos manualmente con reglas de calidad si el resultado final cumple con los resultados esperados en cada columna; como el tipo de datos esperado, si hay nulos, valores únicos y algunas reglas de negocio. Validamos que el resultado final cumple con los requisitos esperados.

In [ ]:
schema = DataFrameSchema({
 "Transaction ID": Column(str, nullable=False, unique=True),
 "Quantity": Column(float, Check.ge(0), nullable=True),
 "Price Per Unit": Column(float, Check.gt(0), nullable=True),
 "Total Spent": Column(float, Check.ge(0), nullable=True),
})
try:
 schema.validate(df, lazy=True)
 print("El dataset cumple con las reglas de calidad definidas.")
except pa.errors.SchemaErrors as err:
 print(err.failure_cases)

In [ ]:
df